# BIOM 200: Microbiome analysis with QIIME 2

Instructors: Rob Knight (robknight@ucsd.edu), Lucas Patel (lpatel@health.ucsd.edu)

The goal is to familiarize yourself with and analyze 16S rRNA amplicon data from Qiita, starting with a feature table and ending with differential abundance. Run the cells in order. Edit only the cells marked **EDIT**. Cells whose code is hidden (click **…** to expand) are definitions; each plot has controls for changing what it shows, and every plot is saved as a PDF in `figures/`. The first part analyzes the class study; the final section, *Your own samples*, repeats the analysis on your group's samples.

## Introduction

The microbes living on and in us shape, and are shaped by, our immune system, our metabolism, and our susceptibility to pathogens. Different body sites host very different communities, and those communities vary far more between people than within a person over time. 16S rRNA amplicon sequencing is the most widely used sequencing modality to ask which bacteria are present in a sample and in what proportions: it is relatively cheap, fast enough to process within a day, and supported by mature, time-tested software. It also has well-known limits. It measures relative rather than absolute abundance, it rarely resolves species, and small studies are easily misled by contamination, sequencing depth, and multiple testing burden. This notebook walks through a standard analysis with those limits in view.

**Read more:** [Gilbert et al. 2018, *Nat Med*](https://doi.org/10.1038/nm.4517) · [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486) · [Knight et al. 2018, *Nat Rev Microbiol*](https://doi.org/10.1038/s41579-018-0029-9)

## Configuring QIIME 2

QIIME 2 is an open-source platform for microbiome analysis. This notebook uses the **QIIME 2 2026.7** distribution. Choose one option.

**Read more:** [Bolyen et al. 2019, *Nat Biotechnol*](https://doi.org/10.1038/s41587-019-0209-9)

### Option A (recommended): GitHub Codespaces

Nothing to install. On the repository page, click **Code → Codespaces → Create codespace on main**. The first start takes a few minutes; QIIME 2 and this notebook open in your browser. When asked for a kernel, choose **QIIME 2 2026.7**. Stop the codespace when you finish (**Codespaces** menu → **Stop Current Codespace**) to save your free monthly hours.

### Option B: install on your own computer

Full instructions: [QIIME 2 installation guide](https://library.qiime2.org/quickstart/amplicon).

1. Install [Miniconda](https://www.anaconda.com/docs/getting-started/miniconda/install). **Windows:** first [install WSL](https://learn.microsoft.com/en-us/windows/wsl/install), then install Miniconda *inside* WSL and follow the Linux instructions.
2. Create the environment for your system (takes 10–30 minutes):

**Linux or Windows (WSL)**
```bash
conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-linux-64-conda.yml
```

**macOS, Apple Silicon (M1 and later)**: runs under Rosetta 2
```bash
CONDA_SUBDIR=osx-64 conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-osx-64-conda.yml
conda activate rachis-qiime2-2026.7
conda config --env --set subdir osx-64
```

**macOS, Intel**
```bash
conda env create --name rachis-qiime2-2026.7 \
  --file https://raw.githubusercontent.com/qiime2/distributions/refs/heads/dev/2026.7/qiime2/released/rachis-qiime2-osx-64-conda.yml
```

3. Add Jupyter (QIIME 2 2026.7 no longer bundles it) and open this notebook:
```bash
conda activate rachis-qiime2-2026.7
conda install -c conda-forge notebook
jupyter notebook biom200_microbiome.ipynb
```

The next cell confirms that the notebook is running inside the QIIME 2 environment.

In [ ]:
%matplotlib inline
import qiime2
print("QIIME 2", qiime2.__version__)

## Loading data from Qiita

Qiita is a web platform that stores microbiome studies together with their metadata and processes every study the same way, so results can be compared across studies. Our reads were amplified from the 16S V4 region with the Earth Microbiome Project protocol, denoised into exact amplicon sequence variants (ASVs) with Deblur, and placed on the Greengenes2 reference tree, which provides both taxonomy and phylogeny.

**Read more:** [Gonzalez et al. 2018, *Nat Methods*](https://doi.org/10.1038/s41592-018-0141-9) · [Thompson et al. 2017, *Nature*](https://doi.org/10.1038/nature24621) · [Amir et al. 2017, *mSystems*](https://doi.org/10.1128/mSystems.00191-16) · [McDonald et al. 2024, *Nat Biotechnol*](https://doi.org/10.1038/s41587-023-01845-1)

For this class exercise, the data will be processed automatically to focus on secondary analysis. To find the input files, open a study on Qiita, and locate each file. Download input files from Qiita to your computer, then put them in the `data/` folder next to this notebook. **In a Codespace**, drag the files onto the `data` folder in the file explorer on the left, or right-click `data` → **Upload...**

| File | What it is | Where to find it in Qiita |
|---|---|---|
| Feature table (`.biom` or `.qza`) | Read counts, samples × ASVs | Study page → data type (16S) → preparation → Deblur feature table → **Download** |
| Sample information (`.tsv`) | One row per sample: body site, participant, sample type, questionnaire | Study page → **Sample Information** → Download |
| Taxonomy (`.qza` or `.tsv`) | Greengenes2 name for each ASV | Publicly available from [Greengenes2](https://ftp.microbio.me/greengenes_release/current/) (`2024.09.taxonomy.asv.tsv.qza`) |
| Phylogeny (`.qza` or `.nwk`) | Pre-constructed phylogenetic tree for UniFrac analyses | Publicly available from [Greengenes2](https://ftp.microbio.me/greengenes_release/current/) (`2024.09.phylogeny.asv.nwk.qza`) |

In [ ]:
# EDIT: paths to the class study files downloaded from Qiita
TABLE = "data/table.biom"            # .biom or .qza
METADATA = "data/sample_information.tsv"
TAXONOMY = "data/taxonomy.qza"       # .qza or .tsv; set to None if the table carries taxonomy
TREE = None                          # optional rooted tree (.qza or .nwk) for Faith's PD and UniFrac

# EDIT: metadata columns
GROUP = "body_site"                  # the main comparison of interest
SUBJECT = "host_subject_id"          # participant identifier
CONTROL_COL = "sample_type"          # column that marks blanks and mock communities
CONTROL_VALUES = ["control blank", "control mock"]

In [ ]:
# Setup: imports, plotting style, and helpers used throughout the notebook
import re
import warnings
from pathlib import Path
from types import SimpleNamespace

import biom
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import skbio
from qiime2 import Artifact, Metadata
from qiime2.plugins import composition, diversity, feature_table, taxa
from scipy import stats
from skbio.stats import subsample_counts
from skbio.stats.distance import permanova

warnings.filterwarnings("ignore", category=FutureWarning)
RESULTS, FIGURES = Path("results"), Path("figures")

# one style for every figure
sns.set_theme(context="notebook", style="ticks", font_scale=0.95,
              rc={"axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110,
                  "axes.titleweight": "bold", "axes.titlesize": 11, "legend.frameon": False,
                  "pdf.fonttype": 42})
SEQ_CMAP = "mako_r"
UP, DOWN, NS = "#4C72B0", "#C44E52", "#BBBBBB"


def save_and_show(fig, ds, name, **params):
    # Save the figure as figures/<dataset>/<name>_<params>.pdf, then display it.
    slug = "_".join(re.sub(r"[^A-Za-z0-9.-]+", "-", f"{k}-{v}") for k, v in params.items())
    out = FIGURES / ds.name
    out.mkdir(parents=True, exist_ok=True)
    fig.savefig(out / f"{name}{'_' + slug if slug else ''}.pdf", bbox_inches="tight")
    plt.show()


def colors_for(ds, col):
    # Fixed colour for each level of a metadata column, identical in every plot.
    levels = sorted(ds.meta[col].astype(str).unique())
    pal = sns.color_palette("deep" if len(levels) <= 10 else "husl", len(levels))
    return dict(zip(levels, pal))


def dropdown(options, value, label):
    return widgets.Dropdown(options=options, value=value if value in options else options[0], description=label,
                            style={"description_width": "initial"})


def slider(lo, hi, value, label, step=1):
    return widgets.IntSlider(min=lo, max=hi, value=value, step=step, description=label, continuous_update=False,
                             style={"description_width": "initial"}, layout={"width": "420px"})


def as_metadata(df):
    df = df.copy()
    df.index.name = "sample-id"
    return Metadata(df)


def short_label(taxon):
    # Last named rank, e.g. 'Streptococcus', or 'Neisseriaceae (f.)' when the genus is unresolved.
    ranks = [r.strip() for r in str(taxon).split(";")]
    named = [r for r in ranks if len(r) > 3 and not r.endswith("__")]
    if not named:
        return "Unassigned"
    if "__" not in named[-1]:
        return named[-1]
    rank, name = named[-1].split("__", 1)
    if rank == "s":
        genus = next((r.split("__", 1)[1] for r in named if r.startswith("g__")), "")
        return name if name.startswith(genus) else f"{genus} {name}".strip()
    return name if rank == "g" else f"{name} ({rank}.)"


def load_table(path):
    if str(path).endswith(".qza"):
        return Artifact.load(path), None
    bt = biom.load_table(path)
    tax = bt.metadata_to_dataframe("observation") if bt.metadata(axis="observation") else None
    return Artifact.import_data("FeatureTable[Frequency]", path, view_type="BIOMV210Format"), tax


def load_taxonomy(path, from_table):
    if path is None:
        if from_table is None:
            raise ValueError("No taxonomy file given and the table carries no taxonomy.")
        cols = [c for c in from_table.columns if c.startswith("taxonomy")]
        df = pd.DataFrame({"Taxon": from_table[cols].astype(str).agg("; ".join, axis=1)})
        df.index.name = "Feature ID"
        return Artifact.import_data("FeatureData[Taxonomy]", df)
    if str(path).endswith(".qza"):
        return Artifact.load(path)
    return Artifact.import_data("FeatureData[Taxonomy]", path, view_type="TSVTaxonomyFormat")


def load_tree(path):
    if str(path).endswith(".qza"):
        return Artifact.load(path)
    return Artifact.import_data("Phylogeny[Rooted]", path)   # Newick (.nwk)


def load_dataset(name, table_path, metadata_path, taxonomy_path=None, tree_path=None, group=None, subject=None,
                 control_col=CONTROL_COL, control_values=CONTROL_VALUES):
    # Load a feature table and its metadata, report the controls, and return a dataset without them.
    table, tax_from_table = load_table(table_path)
    taxonomy = load_taxonomy(taxonomy_path, tax_from_table)
    meta = Metadata.load(metadata_path).to_dataframe()
    counts = table.view(pd.DataFrame)
    shared = counts.index.intersection(meta.index)
    print(f"{name}: {counts.shape[0]} samples and {counts.shape[1]} features; {len(shared)} samples have metadata")
    counts, meta = counts.loc[shared], meta.loc[shared]
    depth = counts.sum(axis=1)
    is_control = meta[control_col].isin(control_values) if control_col in meta else pd.Series(False, index=meta.index)

    tax_series = taxonomy.view(pd.Series)
    for sid in meta.index[is_control]:
        top = counts.loc[sid].sort_values(ascending=False).head(5)
        print(f"  control {sid} ({meta.loc[sid, control_col]}, {int(depth[sid])} reads):",
              ", ".join(f"{short_label(tax_series.get(f, f))} {n:.0f}" for f, n in top[top > 0].items()))

    keep = meta.index[~is_control]
    meta = meta.loc[keep]
    meta = meta.loc[:, meta.notna().any()]
    md = as_metadata(meta)
    table = feature_table.methods.filter_samples(table=table, metadata=md).filtered_table
    tree = load_tree(tree_path) if tree_path else None
    group = group or GROUP
    subject = subject or SUBJECT
    categorical = sorted([c for c in meta.columns if meta[c].dtype == object and 2 <= meta[c].nunique() <= 12],
                         key=lambda c: (c not in (group, subject), c))
    print(f"  {len(keep)} biological samples retained; fields available for grouping: {categorical}")
    return SimpleNamespace(name=name, table=table, taxonomy=taxonomy, tree=tree, meta=meta, md=md,
                           group=group, subject=subject, categorical=categorical)

In [ ]:
study1 = load_dataset("study1", TABLE, METADATA, TAXONOMY, TREE)

### Handling controls

Extraction kits and reagents carry their own trace bacterial DNA, and DNA can leak between neighbouring wells on a plate. In low-biomass samples such as nasal swabs, this contamination can make up a large share of the reads. Blanks reveal what contamination looks like; a mock community of known composition shows whether the pipeline recovers the truth. The cell above listed the most abundant taxa in each control and removed the controls from the dataset used below. Do the blanks look like any of your samples?

**Read more:** [Salter et al. 2014, *BMC Biol*](https://doi.org/10.1186/s12915-014-0087-z) · [Minich et al. 2019, *mSystems*](https://doi.org/10.1128/mSystems.00186-19)

## Taxonomic composition

A stacked bar plot of relative abundance is often the first look at any microbiome dataset: it shows which taxa dominate each sample and whether samples from the same body site resemble each other. Because each bar sums to one, the plot shows proportions only. A taxon whose share rises may have grown, or every other taxon may have shrunk; the data cannot tell these apart.

**Read more:** [Caporaso et al. 2011, *Genome Biol*](https://doi.org/10.1186/gb-2011-12-5-r50) · [Costello et al. 2009, *Science*](https://doi.org/10.1126/science.1177486) · [Gloor et al. 2017, *Front Microbiol*](https://doi.org/10.3389/fmicb.2017.02224)

Change the taxonomic level, the field used to group samples, and the number of taxa shown; tick **Group means** to average samples within each group. Every plot is saved as a PDF in `figures/`, and the interactive QIIME 2 bar plot is saved to `results/`; open `.qzv` files at [view.qiime2.org](https://view.qiime2.org).

In [ ]:
# Composition: collapse to each taxonomic level, then plot
LEVELS = {"Phylum": 2, "Class": 3, "Order": 4, "Family": 5, "Genus": 6}


def add_composition(ds):
    ds.collapsed = {name: taxa.methods.collapse(table=ds.table, taxonomy=ds.taxonomy, level=lvl).collapsed_table
                    for name, lvl in LEVELS.items()}
    ds.rel_at = {}
    for name, art in ds.collapsed.items():
        rel = art.view(pd.DataFrame)
        rel = rel.div(rel.sum(axis=1), axis=0)
        rel.columns = [short_label(c) for c in rel.columns]
        ds.rel_at[name] = rel.T.groupby(level=0).sum().T
    (RESULTS / ds.name).mkdir(parents=True, exist_ok=True)
    taxa.visualizers.barplot(table=ds.table, taxonomy=ds.taxonomy, metadata=ds.md).visualization.save(
        str(RESULTS / ds.name / "taxa-barplot.qzv"))


def plot_composition(ds, level="Genus", group=None, top_n=10, group_means=False):
    group = group or ds.group
    rel = ds.rel_at[level]
    top = rel.mean().sort_values(ascending=False).index[:top_n]
    df = rel[top].assign(Other=1 - rel[top].sum(axis=1))
    grp = ds.meta[group].astype(str)
    if group_means:
        df = df.groupby(grp).mean()
    else:
        order = df.assign(_g=grp, _d=df[top[0]]).sort_values(["_g", "_d"], ascending=[True, False]).index
        df = df.loc[order]
    colors = list(sns.color_palette("tab20", 20)[:top_n]) + [(0.85, 0.85, 0.85)]
    fig, ax = plt.subplots(figsize=(8 if group_means else min(16, max(8, 0.16 * len(df) + 3)), 4.2))
    df.plot.bar(stacked=True, width=0.9 if group_means else 1.0, color=colors, ax=ax, legend=False, linewidth=0)
    ax.set_ylim(0, 1); ax.set_ylabel("Relative abundance"); ax.set_xlabel("")
    if group_means:
        ax.set_xticklabels(df.index, rotation=0)
    else:
        g = grp.loc[df.index].values
        edges = [0] + [i for i in range(1, len(g)) if g[i] != g[i - 1]] + [len(g)]
        ax.set_xticks([(a + b - 1) / 2 for a, b in zip(edges[:-1], edges[1:])])
        ax.set_xticklabels([g[a] for a in edges[:-1]], rotation=0)
        for e in edges[1:-1]:
            ax.axvline(e - 0.5, color="white", lw=2)
    ax.set_title(f"{level}-level composition by {group}")
    ax.legend(*ax.get_legend_handles_labels(), bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, title=level)
    save_and_show(fig, ds, "composition", level=level, by=group, top=top_n, means=group_means)


def explore_composition(ds):
    widgets.interact(plot_composition, ds=widgets.fixed(ds),
                     level=dropdown(list(LEVELS), "Genus", "Level"),
                     group=dropdown(ds.categorical, ds.group, "Group by"),
                     top_n=slider(3, 20, 10, "Taxa shown"),
                     group_means=widgets.Checkbox(value=False, description="Group means"))

In [ ]:
add_composition(study1)
explore_composition(study1)

## Rarefaction

Samples are sequenced to different depths, and deeper samples appear more diverse simply because more reads were drawn. Rarefaction subsamples every sample, without replacement, to a common depth so that diversity is compared on equal terms. It discards data, and whether that is acceptable has been debated; for diversity analyses with very uneven depths it remains a robust choice.

**Read more:** [Weiss et al. 2017, *Microbiome*](https://doi.org/10.1186/s40168-017-0237-y) · [McMurdie & Holmes 2014, *PLoS Comput Biol*](https://doi.org/10.1371/journal.pcbi.1003531)

Move the slider to a depth where most curves have flattened, and note how many samples fall below it and would be dropped. The value left on the slider is used in the next section.

In [ ]:
# Rarefaction: compute curves once, then plot with a depth selector
def add_curves(ds, n_steps=12, iterations=3):
    ds.counts = ds.table.view(pd.DataFrame)
    ds.depth = ds.counts.sum(axis=1)
    steps = np.unique(np.linspace(50, np.percentile(ds.depth, 95), n_steps).astype(int))
    curves = {}
    for sid, row in ds.counts.iterrows():
        v = row.values.astype(int)
        curves[sid] = [np.mean([(subsample_counts(v, d, seed=i) > 0).sum() for i in range(iterations)])
                       if d <= v.sum() else np.nan for d in steps]
    ds.curves = pd.DataFrame(curves, index=steps)


def plot_rarefaction(ds, color_by=None, depth_cutoff=1000):
    color_by = color_by or ds.group
    pal = colors_for(ds, color_by)
    fig, (ax, ax2) = plt.subplots(1, 2, figsize=(11, 4), gridspec_kw={"width_ratios": [3, 2]})
    for sid in ds.curves.columns:
        ax.plot(ds.curves.index, ds.curves[sid], color=pal[str(ds.meta.loc[sid, color_by])], alpha=0.45, lw=1)
    for lvl, c in pal.items():
        ax.plot([], [], color=c, label=lvl)
    ax.axvline(depth_cutoff, color="black", ls="--", lw=1)
    ax.set_xlabel("Reads sampled"); ax.set_ylabel("Observed features"); ax.legend(title=color_by, fontsize=8)
    ax.set_title("Rarefaction curves")
    lost = ds.depth < depth_cutoff
    sns.histplot(x=ds.depth, hue=ds.meta[color_by].astype(str), palette=pal, multiple="stack", bins=25, ax=ax2,
                 legend=False)
    ax2.axvline(depth_cutoff, color="black", ls="--", lw=1)
    ax2.set_xlabel("Reads per sample")
    ax2.set_title(f"{lost.sum()} of {len(ds.depth)} samples below {depth_cutoff:,} reads")
    plt.tight_layout()
    save_and_show(fig, ds, "rarefaction", by=color_by, depth=depth_cutoff)


def explore_rarefaction(ds):
    default = int(max(100, round(ds.depth.quantile(0.1), -2)))
    ds.depth_slider = slider(100, int(ds.depth.max()), default, "Rarefaction depth", step=100)
    widgets.interact(plot_rarefaction, ds=widgets.fixed(ds),
                     color_by=dropdown(ds.categorical, ds.group, "Colour by"), depth_cutoff=ds.depth_slider)

In [ ]:
add_curves(study1)
explore_rarefaction(study1)

## Alpha and beta diversity

Alpha diversity summarizes a single sample: how many taxa it contains (richness), how evenly reads are spread among them (Shannon entropy), or how much of the tree of life they span (Faith's phylogenetic diversity). Beta diversity measures how different two samples are. Bray–Curtis and Jaccard count shared taxa; UniFrac, developed in the Knight lab, also weighs how closely related the unshared taxa are phylogenetically, so two samples dominated by different *Streptococcus* species count as similar.

**Read more:** [Faith 1992, *Biol Conserv*](https://doi.org/10.1016/0006-3207(92)91201-3) · [Lozupone & Knight 2005, *Appl Environ Microbiol*](https://doi.org/10.1128/AEM.71.12.8228-8235.2005) · [Lozupone et al. 2011, *ISME J*](https://doi.org/10.1038/ismej.2010.133)

The next cell rarefies to the depth chosen above and computes the standard metrics. Rerun it after moving the slider. Faith's PD and UniFrac are included when a phylogenetic tree is provided.

In [ ]:
# Diversity: rarefy and compute alpha and beta diversity with QIIME 2
def add_diversity(ds, depth=None):
    ds.rarefaction_depth = depth or ds.depth_slider.value
    lost = ds.depth[ds.depth < ds.rarefaction_depth].index
    print(f"Rarefying to {ds.rarefaction_depth:,} reads; {len(lost)} of {len(ds.depth)} samples dropped: {list(lost)}")
    if ds.tree is not None:
        core = diversity.pipelines.core_metrics_phylogenetic(table=ds.table, phylogeny=ds.tree,
                                                             sampling_depth=ds.rarefaction_depth, metadata=ds.md)
        ds.alpha_metrics = ["observed_features", "shannon", "faith_pd"]
        ds.beta_metrics = ["bray_curtis", "jaccard", "unweighted_unifrac", "weighted_unifrac"]
    else:
        core = diversity.pipelines.core_metrics(table=ds.table, sampling_depth=ds.rarefaction_depth, metadata=ds.md)
        ds.alpha_metrics = ["observed_features", "shannon"]
        ds.beta_metrics = ["bray_curtis", "jaccard"]
    ds.alpha = pd.concat({m: getattr(core, f"{m}_vector").view(pd.Series) for m in ds.alpha_metrics}, axis=1)
    ds.dms = {m: getattr(core, f"{m}_distance_matrix").view(skbio.DistanceMatrix) for m in ds.beta_metrics}
    ds.pcoas = {m: getattr(core, f"{m}_pcoa_results").view(skbio.OrdinationResults) for m in ds.beta_metrics}
    (RESULTS / ds.name).mkdir(parents=True, exist_ok=True)
    for m in ds.beta_metrics:
        getattr(core, f"{m}_emperor").save(str(RESULTS / ds.name / f"{m}-emperor.qzv"))
    return ds.alpha.join(ds.meta[ds.group]).groupby(ds.group).agg(["mean", "std"]).round(2)

In [ ]:
add_diversity(study1)

### Alpha diversity

The test here treats each sample as independent (Mann–Whitney for two groups, Kruskal–Wallis for more). If each participant contributes one sample per group, a paired test such as `scipy.stats.wilcoxon` is more appropriate.

In [ ]:
# Plot: alpha diversity box plot
def plot_alpha(ds, metric="shannon", group=None):
    group = group or ds.group
    df = ds.alpha[[metric]].join(ds.meta[group].astype(str))
    pal = colors_for(ds, group)
    order = sorted(df[group].unique())
    fig, ax = plt.subplots(figsize=(1.3 * len(order) + 2.5, 4))
    sns.boxplot(data=df, x=group, y=metric, order=order, hue=group, palette=pal, fill=False, width=0.55,
                showfliers=False, linewidth=1.4, ax=ax, legend=False)
    sns.stripplot(data=df, x=group, y=metric, order=order, hue=group, palette=pal, size=5, alpha=0.75,
                  jitter=0.15, ax=ax, legend=False)
    groups = [g[metric].values for _, g in df.groupby(group)]
    test = stats.mannwhitneyu(*groups) if len(groups) == 2 else stats.kruskal(*groups)
    name = "Mann–Whitney" if len(groups) == 2 else "Kruskal–Wallis"
    ax.set_title(f"{metric.replace('_', ' ')} by {group}\n{name} p = {test.pvalue:.3g}")
    ax.set_xlabel(""); ax.set_ylabel(metric.replace("_", " "))
    save_and_show(fig, ds, "alpha", metric=metric, by=group)


def explore_alpha(ds):
    widgets.interact(plot_alpha, ds=widgets.fixed(ds), metric=dropdown(ds.alpha_metrics, "shannon", "Metric"),
                     group=dropdown(ds.categorical, ds.group, "Group by"))

In [ ]:
explore_alpha(study1)

### Beta diversity

Every cell is the distance between two samples, ordered by group; lighter means more similar. Light blocks along the diagonal mean that samples within a group resemble each other. The title compares the mean distance within groups with the mean distance between groups; for technical replicates, the within-group distance is the noise floor.

In [ ]:
# Plot: distance-matrix heatmap ordered by group
def plot_distance_matrix(ds, metric="bray_curtis", group=None):
    group = group or ds.group
    dm = ds.dms[metric]
    ids = list(dm.ids)
    grp = ds.meta.loc[ids, group].astype(str)
    order = grp.sort_values(kind="stable").index
    df = dm.to_data_frame().loc[order, order]
    pal = colors_for(ds, group)
    strip = list(grp.loc[order].map(pal))
    same = grp.values[:, None] == grp.values[None, :]
    off = ~np.eye(len(ids), dtype=bool)
    within, between = dm.data[same & off].mean(), dm.data[~same].mean()
    g = sns.clustermap(df, row_cluster=False, col_cluster=False, row_colors=strip, col_colors=strip, cmap=SEQ_CMAP,
                       xticklabels=False, yticklabels=False, figsize=(6.5, 6.5), dendrogram_ratio=0.02,
                       colors_ratio=0.025, cbar_pos=(0.3, -0.02, 0.45, 0.022),
                       cbar_kws={"orientation": "horizontal", "label": f"{metric.replace('_', ' ')} distance"})
    g.ax_heatmap.set_xlabel(""); g.ax_heatmap.set_ylabel("")
    for lvl, c in pal.items():
        g.ax_heatmap.bar(0, 0, color=c, label=lvl)
    g.ax_heatmap.legend(title=group, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    g.fig.suptitle(f"{metric.replace('_', ' ')}: mean within-group {within:.2f}, between-group {between:.2f}",
                   y=1.02, fontweight="bold", fontsize=11)
    save_and_show(g.fig, ds, "distance-matrix", metric=metric, by=group)


def explore_distance_matrix(ds):
    widgets.interact(plot_distance_matrix, ds=widgets.fixed(ds),
                     metric=dropdown(ds.beta_metrics, "bray_curtis", "Metric"),
                     group=dropdown(ds.categorical, ds.group, "Group by"))

In [ ]:
explore_distance_matrix(study1)

## Ordination (PCoA) and PERMANOVA

A distance matrix with dozens of samples is hard to read directly. Principal coordinates analysis (PCoA) projects it into a few axes that capture as much of the variation as possible; each axis label gives the share of variation it explains. PERMANOVA then asks whether samples in different groups are more different from each other than samples within a group, using permutations rather than distributional assumptions. It is also sensitive to differences in spread between groups, which `skbio.stats.distance.permdisp` tests. EMPeror, the Knight lab's interactive 3D viewer, is saved to `results/` for each metric.

**Read more:** [Anderson 2001, *Austral Ecol*](https://doi.org/10.1111/j.1442-9993.2001.01070.pp.x) · [Vázquez-Baeza et al. 2013, *GigaScience*](https://doi.org/10.1186/2047-217X-2-16)

In [ ]:
# Plot: PCoA scatter with PERMANOVA for the selected field
def run_permanova(ds, metric, column):
    dm = ds.dms[metric]
    grouping = ds.meta.loc[list(dm.ids), column].astype(str)
    if grouping.nunique() < 2:
        return None
    r = permanova(dm, grouping, permutations=999)
    return r["test statistic"], r["p-value"]


def permanova_table(ds):
    rows = [{"metric": m, "grouping": c, "pseudo-F": round(r[0], 2), "p": r[1]}
            for m in ds.beta_metrics for c in [ds.group, ds.subject]
            if c in ds.meta and (r := run_permanova(ds, m, c))]
    return pd.DataFrame(rows)


def plot_pcoa(ds, metric="bray_curtis", color_by=None, axes="PC1 vs PC2"):
    color_by = color_by or ds.group
    pc = ds.pcoas[metric]
    i, j = [int(a[2:]) - 1 for a in axes.split(" vs ")]
    coords = pc.samples.iloc[:, [i, j]].join(ds.meta[color_by].astype(str))
    expl = pc.proportion_explained * 100
    pal = colors_for(ds, color_by)
    fig, ax = plt.subplots(figsize=(6.2, 5))
    sns.scatterplot(data=coords, x=coords.columns[0], y=coords.columns[1], hue=color_by, palette=pal, s=55,
                    alpha=0.85, edgecolor="white", linewidth=0.5, ax=ax)
    ax.set_xlabel(f"PC{i + 1} ({expl.iloc[i]:.0f}%)"); ax.set_ylabel(f"PC{j + 1} ({expl.iloc[j]:.0f}%)")
    ax.set_xticks([]); ax.set_yticks([])
    res = run_permanova(ds, metric, color_by)
    sub = f"PERMANOVA by {color_by}: pseudo-F = {res[0]:.2f}, p = {res[1]:.3g}" if res else ""
    ax.set_title(f"{metric.replace('_', ' ')} PCoA\n{sub}")
    ax.legend(title=color_by, bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
    save_and_show(fig, ds, "pcoa", metric=metric, by=color_by, axes=axes.replace(" vs ", "-"))


def explore_pcoa(ds):
    widgets.interact(plot_pcoa, ds=widgets.fixed(ds), metric=dropdown(ds.beta_metrics, "bray_curtis", "Metric"),
                     color_by=dropdown(ds.categorical, ds.group, "Colour by"),
                     axes=dropdown(["PC1 vs PC2", "PC1 vs PC3", "PC2 vs PC3"], "PC1 vs PC2", "Axes"))

In [ ]:
explore_pcoa(study1)

PERMANOVA for every metric, by the main comparison and by participant:

In [ ]:
permanova_table(study1)

## Differential abundance

Which taxa differ between groups? Because 16S data are compositional, an increase in one taxon makes every other taxon look smaller, so ordinary per-taxon tests on relative abundances can report changes that never happened. Methods such as ANCOM-BC estimate and correct for this bias, and reference frames make clear that only *ratios* between taxa are truly measured. Even so, different methods applied to the same data disagree substantially, so treat any result from a small study as a hypothesis rather than a finding.

**Read more:** [Morton et al. 2019, *Nat Commun*](https://doi.org/10.1038/s41467-019-10656-5) · [Lin & Peddada 2020, *Nat Commun*](https://doi.org/10.1038/s41467-020-17041-7) · [Nearing et al. 2022, *Nat Commun*](https://doi.org/10.1038/s41467-022-28034-z)

Genera present in fewer than `min_prevalence` of samples are filtered out first, since they cannot be tested meaningfully. `reference` sets the group every other group is compared with (default: first alphabetically).

In [ ]:
# Differential abundance: ANCOM-BC on genus-level counts, then a forest plot
def add_differential_abundance(ds, min_prevalence=0.25, reference=None):
    genus = ds.collapsed["Genus"]
    g = genus.view(pd.DataFrame)
    genus_f = feature_table.methods.filter_features(
        table=genus, min_samples=int(np.ceil(min_prevalence * len(g)))).filtered_table
    print(f"Testing {genus_f.view(pd.DataFrame).shape[1]} of {g.shape[1]} genera")
    kwargs = {"reference_levels": [f"{ds.group}::{reference}"]} if reference else {}
    res = composition.methods.ancombc(table=genus_f, metadata=ds.md, formula=ds.group, **kwargs)
    out = RESULTS / ds.name
    out.mkdir(parents=True, exist_ok=True)
    composition.visualizers.da_barplot(data=res.differentials).visualization.save(str(out / "ancombc-barplot.qzv"))
    res.differentials.export_data(out / "ancombc")
    part = {k: pd.read_csv(out / "ancombc" / f"{k}_slice.csv", index_col="id").drop(columns="(Intercept)")
            for k in ["lfc", "se", "q_val"]}
    da = pd.concat({k: v.stack() for k, v in part.items()}, axis=1).reset_index()
    da.columns = ["taxon", "comparison", "log_fold_change", "se", "q"]
    ref = reference or sorted(ds.meta[ds.group].astype(str).unique())[0]
    da["taxon"] = da["taxon"].map(short_label)
    da["comparison"] = da["comparison"].str.replace(ds.group, "", regex=False) + f" vs {ref}"
    dup = da.groupby("comparison")["taxon"].transform(lambda t: t.groupby(t).cumcount())
    da.loc[dup > 0, "taxon"] += " #" + (dup[dup > 0] + 1).astype(str)   # distinct lineages sharing a short name
    ds.da = da
    print(f"{(da.q < 0.05).sum()} genus–comparison pairs with q < 0.05 (Holm-adjusted)")


def plot_forest(ds, comparison, top_n=15, q_max=0.05):
    df = ds.da[ds.da["comparison"] == comparison]
    df = df.reindex(df["log_fold_change"].abs().sort_values(ascending=False).index).head(top_n)
    df = df.sort_values("log_fold_change")
    sig = df["q"] < q_max
    color = np.where(~sig, NS, np.where(df["log_fold_change"] > 0, UP, DOWN))
    fig, ax = plt.subplots(figsize=(6.5, 0.32 * len(df) + 1.4))
    y = np.arange(len(df))
    ax.hlines(y, df["log_fold_change"] - 1.96 * df["se"], df["log_fold_change"] + 1.96 * df["se"], color=color, lw=1.6)
    ax.scatter(df["log_fold_change"], y, color=color, s=36, zorder=3)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(y); ax.set_yticklabels(df["taxon"])
    ax.set_xlabel("Log-fold change (ln), 95% CI"); ax.set_ylabel("")
    ax.set_title(f"{comparison}: {sig.sum()} of {len(df)} shown with q < {q_max:g}")
    save_and_show(fig, ds, "forest", comparison=comparison, top=top_n, q=f"{q_max:g}")


def explore_forest(ds):
    comparisons = sorted(ds.da["comparison"].unique())
    widgets.interact(plot_forest, ds=widgets.fixed(ds), comparison=dropdown(comparisons, comparisons[0], "Comparison"),
                     top_n=slider(5, 40, 15, "Genera shown"),
                     q_max=widgets.FloatLogSlider(value=0.05, base=10, min=-4, max=0, step=0.25, description="q cut-off",
                                                  continuous_update=False, style={"description_width": "initial"}))

In [ ]:
add_differential_abundance(study1, min_prevalence=0.25, reference=None)
explore_forest(study1)

Questions to consider:

- How many genera were tested, and is the reported q-value corrected for that number?
- Is the genus abundant enough in the bar plot to be a plausible biological difference, or is it rare?
- Does it also appear in the blanks?

## Your own samples

Now repeat the analysis on the samples designed and collected by the group. Every step reuses the functions defined above, so each cell is a single line. Download your group's feature table and sample information from Qiita, upload them to `data/`, and **EDIT** the cell below. Set `MY_GROUP` to the metadata field that encodes your comparison (for example, before vs. after brushing).

Before interpreting anything, check two things: that your blanks are low and unlike your samples, and that each group has enough samples left after rarefaction. With a handful of samples per group, treat every p-value as a description of these samples, not of people in general.

In [ ]:
# EDIT: your group's files and comparison
MY_TABLE = "data/my_table.biom"
MY_METADATA = "data/my_sample_information.tsv"
MY_TAXONOMY = "data/taxonomy.qza"      # same Greengenes2 taxonomy as above, or None if the table carries it
MY_TREE = TREE
MY_GROUP = "body_site"                 # the field that encodes your comparison
MY_SUBJECT = SUBJECT

mine = load_dataset("my_samples", MY_TABLE, MY_METADATA, MY_TAXONOMY, MY_TREE, group=MY_GROUP, subject=MY_SUBJECT)

### Composition

In [ ]:
add_composition(mine)
explore_composition(mine)

### Rarefaction

Choose a depth, then run the next cell.

In [ ]:
add_curves(mine)
explore_rarefaction(mine)

### Alpha and beta diversity

In [ ]:
add_diversity(mine)

In [ ]:
explore_alpha(mine)

In [ ]:
explore_distance_matrix(mine)

In [ ]:
explore_pcoa(mine)

In [ ]:
permanova_table(mine)

### Differential abundance

In [ ]:
add_differential_abundance(mine, min_prevalence=0.25, reference=None)
explore_forest(mine)

Compare your results with your pre-registered prediction. Which effects are larger than the differences between technical replicates, and which would you defend in a paper?